# Lesson 03 - एजेंटिक डिज़ाइन पैटर्न

इस पाठ में, हम प्रभावी AI एजेंट बनाने के लिए तीन मौलिक डिज़ाइन पैटर्न का अन्वेषण करते हैं:

1. **स्पष्ट एजेंट निर्देश** — एजेंट के व्यवहार का मार्गदर्शन करने वाले सटीक, भूमिका-परिभाषित प्रॉम्प्ट तैयार करना  
2. **Pydantic मॉडल के साथ संरचित आउटपुट** — यह सुनिश्चित करना कि एजेंट प्रत्याशित, मान्य डेटा वापस करें  
3. **एकल उत्तरदायित्व एजेंट** — केंद्रित एजेंट डिजाइन करना जो हर एक अच्छी तरह से एक काम करता है  

हम प्रत्येक पैटर्न को एक **यात्रा गंतव्य अनुशंषाकर्ता** परिदृश्य पर लागू करेंगे, जो क्रमशः एक ऐसा सिस्टम बनाएगा जो गंतव्य सुझा सके, उपलब्धता जांच सके, और लॉजिस्टिक्स संभाल सके।


## सेटअप


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## पैटर्न 1: स्पष्ट एजेंट निर्देश

सबसे प्रभावशाली पैटर्न भी सबसे सरल है: अपने एजेंट के लिए स्पष्ट, विस्तृत निर्देश लिखना।

अच्छे निर्देश परिभाषित करते हैं:
- **कौन** एजेंट है (व्यक्तित्व और स्वर)
- **क्या** इसे करना चाहिए (चरण-दर-चरण जिम्मेदारियां)
- **कैसे** इसे व्यवहार करना चाहिए (सीमाएं और शैली)

नीचे, हम एक ट्रैवल कंसिएज एजेंट बनाते हैं जिसमें स्पष्ट निर्देश होते हैं जो प्रत्येक प्रतिक्रिया को आकार देते हैं जो यह उत्पन्न करता है।


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Pattern 2: Pydantic मॉडल्स के साथ संरचित आउटपुट

फ्री-फॉर्म टेक्स्ट बातचीत के लिए उपयोगी है, लेकिन डाउनस्ट्रीम सिस्टम्स को संरचित डेटा की आवश्यकता होती है।  
**Pydantic मॉडल्स** को एक **टूल फ़ंक्शन** के साथ जोड़कर, हम:

- एजेंट के आउटपुट के लिए एक सटीक स्कीमा परिभाषित कर सकते हैं  
- प्रतिक्रियाओं को स्वचालित रूप से सत्यापित कर सकते हैं  
- एप्लिकेशन लॉजिक में एजेंट के परिणामों को विश्वसनीय रूप से एकीकृत कर सकते हैं  

हम एक ऐसा टूल भी प्रस्तुत करते हैं जो गंतव्य विवरण लौटाता है ताकि एजेंट अपनी सिफारिशों को वास्तविक डेटा में आधार कर सके।


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## पैटर्न 3: एकल जिम्मेदारी एजेंट्स

जटिल कार्यों का लाभ कई केंद्रित एजेंट्स में काम विभाजित करने से होता है, जिनमें से प्रत्येक की एकल जिम्मेदारी होती है:

- एक **डेस्टिनेशन एक्सपर्ट** जो स्थानों और उपलब्धता के बारे में जानता है
- एक **लॉजिस्टिक्स प्लानर** जो उड़ानों, होटलों, और यात्रा कार्यक्रमों को संभालता है

यह सॉफ़्टवेयर इंजीनियरिंग के *सेपरेशन ऑफ कॉन्सर्न्स* सिद्धांत को दर्शाता है — प्रत्येक एजेंट को स्वतंत्र रूप से परीक्षण, रखरखाव और सुधार करना आसान होता है।


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## सारांश

इस पाठ में हमने एक यात्रा अनुशंसक परिदृश्य में तीन एजेंटिक डिज़ाइन पैटर्न लागू किए:

| पैटर्न | मुख्य विचार | लाभ |
|---|---|---|
| **स्पष्ट निर्देश** | पर्सोना, जिम्मेदारियां, और सीमाओं को पहले से परिभाषित करें | सुसंगत, ब्रांड के अनुरूप एजेंट व्यवहार |
| **संरचित आउटपुट** | प्रतिक्रिया प्रारूप के रूप में Pydantic मॉडल का उपयोग करें | सत्यापित, मशीन-पठनीय परिणाम |
| **एकल जिम्मेदारी** | प्रत्येक एजेंट को एक केंद्रित काम दें | परीक्षण, रखरखाव, और संयोजन में आसान |

ये पैटर्न सहज रूप से मिलकर कार्य करते हैं — आप एकल-जिम्मेदारी एजेंट के अंदर स्पष्ट निर्देशों को संरचित आउटपुट के साथ जोड़ सकते हैं ताकि मजबूत, उत्पादन-तैयार सिस्टम बनाए जा सकें।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयास करते हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में ही प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
